In [ ]:
import json
import os
import re
import time
from datetime import datetime
from urllib.parse import quote

import requests


FRAGMENT_FILE = "fragment_final.jsonl"   
TON_FILE = "ton.jsonl"                

TONAPI_KEY = "InsertYourKey"

MAX_NEW_OWNERS = None        
SLEEP_BETWEEN_CALLS = 0.25


TONAPI_BASE = "https://tonapi.io"


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def read_jsonl(path):
    if not os.path.exists(path):
        return []
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")



def is_wallet_address(s):
    if not s:
        return False
    s = str(s).strip()

    # (EQ... / UQ...)
    if re.fullmatch(r"[EU]Q[A-Za-z0-9_-]{40,}", s):
        return True

    # (0:... oppure -1:...)
    if re.fullmatch(r"-?\d+:[0-9a-fA-F]{64}", s):
        return True

    return False


def normalize_owner_for_dns(owner):
    if not owner:
        return None

    owner = str(owner).strip()

    if is_wallet_address(owner):
        return None

    if owner.startswith("@") and len(owner) > 1:
        return owner[1:] + ".t.me"

    m = re.search(r"(?:https?://)?t\.me/([A-Za-z0-9_]{3,})", owner)
    if m:
        return m.group(1) + ".t.me"

    if owner.endswith(".t.me") or owner.endswith(".ton"):
        return owner

   
    return owner



def owners_from_fragment(fragment_records):
    owners = set()
    for rec in fragment_records:
        history = rec.get("Ownership history") or []
        for h in history:
            raw_owner = (h or {}).get("Owner")
            norm = normalize_owner_for_dns(raw_owner)
            if norm:
                owners.add(norm)
            elif is_wallet_address(raw_owner):
                owners.add(str(raw_owner).strip())
    return sorted(list(owners))


def known_owners_from_ton(ton_records):
    s = set()
    for r in ton_records:
        o = r.get("owner")
        if isinstance(o, str) and o:
            s.add(o)
    return s

def find_address(obj):
    """
    Cerca un address dentro JSON TonAPI (best effort).
    """
    if isinstance(obj, dict):
        for v in obj.values():
            a = find_address(v)
            if a:
                return a
    elif isinstance(obj, list):
        for it in obj:
            a = find_address(it)
            if a:
                return a
    elif isinstance(obj, str):
        if re.fullmatch(r"-?\d+:[0-9a-fA-F]{64}", obj):
            return obj
        if re.fullmatch(r"[EU]Q[A-Za-z0-9_-]{40,}", obj):
            return obj
    return None


def dns_resolve(session, dns_name):
    """
    DNS resolve SOLO per name.t.me / name.ton ecc.
    Se TonAPI risponde 400/404 -> ritorna None senza crash.
    """
    url = f"{TONAPI_BASE}/v2/dns/{quote(dns_name, safe='')}/resolve"
    r = session.get(url, timeout=30)

    if r.status_code in (400, 404):
        return None

    if r.status_code >= 400:
        return None

    try:
        return find_address(r.json())
    except Exception:
        return None


def account_nfts(session, address):
    url = f"{TONAPI_BASE}/v2/accounts/{quote(address, safe='')}/nfts"
    r = session.get(url, params={"limit": 1000, "offset": 0}, timeout=30)

    if r.status_code == 404:
        return []

    if r.status_code >= 400:
        return []

    data = r.json()
    items = data.get("nft_items") or data.get("items") or []
    return items if isinstance(items, list) else []


def filter_usernames(nft_items):
    """
    Tiene solo NFT con metadata.name che inizia con '@'
    """
    out = []
    for it in nft_items:
        md = it.get("metadata") or {}
        name = md.get("name")
        if isinstance(name, str) and name.startswith("@"):
            out.append(name)
            
    seen = set()
    uniq = []
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq


def main():
    if not TONAPI_KEY or "INSERISCI" in TONAPI_KEY:
        print("ERRORE: inserisci la TONAPI_KEY.")
        return

    if not os.path.exists(FRAGMENT_FILE):
        print("ERRORE: non trovo", FRAGMENT_FILE)
        return

    fragment_records = read_jsonl(FRAGMENT_FILE)
    ton_records = read_jsonl(TON_FILE)

    owners_all = owners_from_fragment(fragment_records)
    known = known_owners_from_ton(ton_records)

    new_owners = [o for o in owners_all if o not in known]
    if MAX_NEW_OWNERS is not None:
        new_owners = new_owners[:MAX_NEW_OWNERS]

    if not new_owners:
        print("Nessun nuovo owner da processare.")
        return

    session = requests.Session()
    session.headers.update({
        "Authorization": "Bearer " + TONAPI_KEY,
        "Accept": "application/json",
        "User-Agent": "ThesisTonExtractor/1.0",
    })

    total = len(new_owners)
    for i, owner in enumerate(new_owners, start=1):
        # Caso 1: owner è già un address
        if is_wallet_address(owner):
            addr = owner
        else:
            # Caso 2: owner è DNS -> resolve
            addr = dns_resolve(session, owner)

        if not addr:
            append_jsonl(TON_FILE, {
                "owner": owner,
                "raw_address": None,
                "nft_posseduti": [],
                "scrape_date": now_str(),
                "note": "resolve_failed",
            })
            print(f"[{i}/{total}] resolve fail: {owner}")
            continue

        nfts = account_nfts(session, addr)
        usernames = filter_usernames(nfts)

        append_jsonl(TON_FILE, {
            "owner": owner,
            "raw_address": addr,
            "nft_posseduti": usernames,
            "scrape_date": now_str(),
        })

        print(f"[{i}/{total}] OK: {owner} -> {len(usernames)} username")
        time.sleep(SLEEP_BETWEEN_CALLS)


if __name__ == "__main__":
    main()

Nessun nuovo owner da processare.


In [37]:
import pandas as pd
df = pd.read_json("ton.jsonl", lines=True)
df

,owner,raw_address,nft_posseduti,scrape_date,note
0,UQA3KfvESC9n21AggYvk7BVjCRTOVJ_fbXO9c5ommympy8tk,UQA3KfvESC9n21AggYvk7BVjCRTOVJ_fbXO9c5ommympy8tk,[],2026-01-16 16:52:56,NaN
1,UQADYAyrsTz2upImBHtU82Y0qKgS5BYPW2j69VsYbH5AjbFX,UQADYAyrsTz2upImBHtU82Y0qKgS5BYPW2j69VsYbH5AjbFX,[],2026-01-16 16:52:57,NaN
2,UQAIFunALREOeQ99syMbO6sSzM_Fa1RsPD5TBoS0qVeKQ73U,UQAIFunALREOeQ99syMbO6sSzM_Fa1RsPD5TBoS0qVeKQ73U,"[@inagent, @samodelkin]",2026-01-16 16:52:59,NaN
3,UQAJ_outPhRzV1DdF2xnoegaS2Gqdmqyk8DIcjidno0xsdHY,UQAJ_outPhRzV1DdF2xnoegaS2Gqdmqyk8DIcjidno0xsdHY,"[@luxeapp, @aquapizza, @id602, @fragment1688, ...",2026-01-16 16:53:00,NaN
4,UQAQa5ua61tv4XLi16xq5T_672vu72VENN415BXUYy17G3jH,UQAQa5ua61tv4XLi16xq5T_672vu72VENN415BXUYy17G3jH,"[@postmodern, @postscriptum, @irano, @stabby, ...",2026-01-16 16:53:03,NaN
...,...,...,...,...,...
6984,ww8888.t.me,None,[],2026-01-17 15:02:38,resolve_failed
6985,xbanking.ton,None,[],2026-01-17 15:02:38,resolve_failed
6986,xx_xx.t.me,None,[],2026-01-17 15:02:38,resolve_failed
6987,yewu.ton,None,[],2026-01-17 15:02:38,resolve_failed
